In [1]:
!pip -q install chromadb sentence-transformers langchain-chroma langchain-huggingface

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 840.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 90.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently 

In [2]:
import chromadb

from sentence_transformers import SentenceTransformer

from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

In [3]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Create a dataset designed to show the problem

In [4]:
documents = [
    "Python generators produce values lazily instead of creating all values in memory at once.",

    "Generators produce values one at a time using lazy evaluation.",

    "Python generator functions use the yield keyword to pause execution and resume later.",

    "Generators are useful for processing large datasets because they reduce memory usage.",

    "Generator expressions provide a concise way to create generators that produce values lazily.",

    "FastAPI is a Python framework for building web APIs.",

    "Redis is an in-memory data store commonly used for caching and queues.",

    "Docker packages applications and dependencies into containers."
]

Create embeddings

In [5]:
embeddings = embedding_model.encode(documents)

print("Documents:", len(documents))
print("Dimensions:", len(embeddings[0]))

Documents: 8
Dimensions: 384


Create Chroma

In [6]:
client = chromadb.Client()

collection = client.get_or_create_collection(
    name="day9_retrieval"
)

Store them

In [7]:
collection.add(
    ids=[f"doc_{i}" for i in range(len(documents))],
    documents=documents,
    embeddings=embeddings.tolist()
)

Similarity search

In [8]:
query = "How do Python generators help with memory?"

In [9]:
query_embedding = embedding_model.encode(
    [query]
)[0]

In [10]:
results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=5,
    include=["documents", "distances"]
)

In [11]:
for i, (doc, distance) in enumerate(
    zip(
        results["documents"][0],
        results["distances"][0]
    ),
    start=1
):
    print(f"{i}. Distance: {distance:.4f}")
    print(doc)
    print()

1. Distance: 0.5298
Python generators produce values lazily instead of creating all values in memory at once.

2. Distance: 0.6184
Generators are useful for processing large datasets because they reduce memory usage.

3. Distance: 0.8291
Python generator functions use the yield keyword to pause execution and resume later.

4. Distance: 1.0113
Generators produce values one at a time using lazy evaluation.

5. Distance: 1.0113
Generator expressions provide a concise way to create generators that produce values lazily.



Now introduce MMR

In [12]:
hf_embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [13]:
vectorstore = Chroma(
    client=client,
    collection_name="day9_retrieval",
    embedding_function=hf_embeddings
)

In [14]:
similarity_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 3
    }
)

In [15]:
similarity_docs = similarity_retriever.invoke(
    query
)

In [16]:
print("SIMILARITY SEARCH\n")

for i, doc in enumerate(similarity_docs, start=1):
    print(f"{i}. {doc.page_content}\n")

SIMILARITY SEARCH

1. Python generators produce values lazily instead of creating all values in memory at once.

2. Generators are useful for processing large datasets because they reduce memory usage.

3. Python generator functions use the yield keyword to pause execution and resume later.



In [17]:
mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 3,
        "fetch_k": 6,
        "lambda_mult": 0.5
    }
)

In [18]:
mmr_docs = mmr_retriever.invoke(
    query
)

In [19]:
print("MMR SEARCH\n")

for i, doc in enumerate(mmr_docs, start=1):
    print(f"{i}. {doc.page_content}\n")

MMR SEARCH

1. Python generators produce values lazily instead of creating all values in memory at once.

2. Generators are useful for processing large datasets because they reduce memory usage.

3. Redis is an in-memory data store commonly used for caching and queues.



STEP 15 — Experiment with λ

In [20]:
for lambda_value in [0.0, 0.25, 0.5, 0.75, 1.0]:

    retriever = vectorstore.as_retriever(
        search_type="mmr",
        search_kwargs={
            "k": 3,
            "fetch_k": 6,
            "lambda_mult": lambda_value
        }
    )

    docs = retriever.invoke(query)

    print("=" * 70)
    print(f"LAMBDA = {lambda_value}")
    print("=" * 70)

    for i, doc in enumerate(docs, start=1):
        print(f"{i}. {doc.page_content}")

LAMBDA = 0.0
1. Python generators produce values lazily instead of creating all values in memory at once.
2. Generators are useful for processing large datasets because they reduce memory usage.
3. Redis is an in-memory data store commonly used for caching and queues.
LAMBDA = 0.25
1. Python generators produce values lazily instead of creating all values in memory at once.
2. Generators are useful for processing large datasets because they reduce memory usage.
3. Redis is an in-memory data store commonly used for caching and queues.
LAMBDA = 0.5
1. Python generators produce values lazily instead of creating all values in memory at once.
2. Generators are useful for processing large datasets because they reduce memory usage.
3. Redis is an in-memory data store commonly used for caching and queues.
LAMBDA = 0.75
1. Python generators produce values lazily instead of creating all values in memory at once.
2. Generators are useful for processing large datasets because they reduce memory usa

Compare the two strategies

In [21]:
print("SIMILARITY RESULTS")
print("=" * 60)

for i, doc in enumerate(similarity_docs, start=1):
    print(f"{i}. {doc.page_content}")

print("\n\nMMR RESULTS")
print("=" * 60)

for i, doc in enumerate(mmr_docs, start=1):
    print(f"{i}. {doc.page_content}")

SIMILARITY RESULTS
1. Python generators produce values lazily instead of creating all values in memory at once.
2. Generators are useful for processing large datasets because they reduce memory usage.
3. Python generator functions use the yield keyword to pause execution and resume later.


MMR RESULTS
1. Python generators produce values lazily instead of creating all values in memory at once.
2. Generators are useful for processing large datasets because they reduce memory usage.
3. Redis is an in-memory data store commonly used for caching and queues.


# Day 9 Reflection

## What I learned

- Retrieval strategy determines which information is passed to the LLM.
- Basic similarity search retrieves documents primarily based on similarity
  to the query.
- MMR (Maximal Marginal Relevance) balances relevance with diversity.
- MMR can reduce redundant retrieved information.
- `k` controls the final number of documents returned.
- `fetch_k` controls the candidate pool considered by MMR.
- `lambda_mult` controls the relevance-versus-diversity trade-off.
- A higher lambda generally prioritizes relevance.
- A lower lambda generally prioritizes diversity.
- MMR is not automatically better than similarity search.
- Retrieval strategies should be evaluated using representative queries and
  downstream answer quality.

## Key takeaway

Retrieval is a design decision, not just a database operation.

The goal is not simply to retrieve the most similar documents.
The goal is to provide the LLM with useful, relevant, and sufficiently
diverse context for answering the user's question.